# Glutamate response analysis run

Use this notebook to load one session, run activation/tuning/sequence analyses, and save standardized output tables.

In [1]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    resolve_glutamate_analysis_paths,
    run_glutamate_analysis,
)

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
target_mice = [
    803496,
    804730,
    804733,810196,
    809047,803121,
    826033,838410,834788
]

In [4]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [5]:
assets[0]

SessionAssets(session_id='803496_2025-07-25_13-02-10', subject_id=803496, session_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496'), summary_mat=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08/source_extraction/ExperimentSummary/SummaryLoCo-260117-185357.mat'), bonsai_event_log_csv=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp/bonsai_event_log.csv'), harp_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp'), photodiode_pkl=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp/extracted_files

In [ ]:
for asset in assets[:]:
    session_root = asset.session_dir
    paths = resolve_glutamate_analysis_paths(session_root)
    print(paths)
    
    config = GlutamateAnalysisConfig(
        tuning_method="hybrid",          # or "manova"
        manova_stat="Wilks' lambda",
        manova_max_timepoints=10,        # reduce from 20
        manova_use_post_only=True,       # much faster, focuses on evoked period
        random_seed=0,
    )

    results = run_glutamate_analysis(session_root, config=config)
    
    {k: v.shape for k, v in results.items() if hasattr(v, 'shape')}
    
    activation_summary = results['activation_summary_table']
    tuning_summary = results['tuning_summary_table']
    sequence_summary = results['sequence_summary_table']

    display(activation_summary.head())
    display(tuning_summary.head())
    display(sequence_summary.head())
    
    output_dir = resolve_glutamate_analysis_paths(session_root).output_dir
    print(output_dir)
    print(sorted(p.name for p in output_dir.iterdir()))

GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_sequence_df.npz'), output_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate_analysis'))
